# Phase 4 — Feature engineering et pipeline

**Objectif :** transformer les données nettoyées en entrées exploitables par un modèle, en traduisant les insights de l'EDA en variables, **sans aucune fuite de données**.

**Plan**
1. Découpage train/test (avant toute transformation)
2. Features métier
3. Prétraitement et pipeline scikit-learn
4. Validation des choix par validation croisée

> **Règle d'or :** le jeu de test est mis sous clé dès la section 1. Toutes les décisions de ce notebook sont prises **uniquement sur le jeu d'entraînement**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from churn.config import TARGET, RANDOM_STATE
from churn.data import load_clean
from churn.features import split_data
from churn.viz import set_style, save_fig, fmt_pct, fmt_int, BLUE, ORANGE, TEXT, TEXT_SECONDARY, GRID

set_style()
df = load_clean()

## 1. Découpage train/test stratifié

- **80 % entraînement / 20 % test**, avec un découpage **stratifié** (même proportion de churners dans chaque jeu).
- Le découpage est fait **avant** toute transformation : si on standardisait les données avant de découper, la moyenne et l'écart-type incluraient les clients de test, et l'évaluation serait biaisée (*data leakage*).

In [2]:
X_train, X_test, y_train, y_test = split_data(df)

summary = pd.DataFrame({
    "clients": [len(X_train), len(X_test), len(df)],
    "churners": [y_train.sum(), y_test.sum(), df[TARGET].sum()],
    "taux_churn": [y_train.mean(), y_test.mean(), df[TARGET].mean()],
}, index=["Entraînement", "Test", "Total"])
summary.style.format({"taux_churn": "{:.2%}"})

,clients,churners,taux_churn
Entraînement,5634,1495,26.54%
Test,1409,374,26.54%
Total,7043,1869,26.54%


**Constat :** 5 634 clients d'entraînement et 1 409 de test, avec un taux de churn identique (~26,5 %) dans les deux jeux. Le jeu de test n'est **plus utilisé avant la Phase 6**.